Import Library

In [1]:
import os, time
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import json

Konfigurasi

In [2]:
df_train_meta = pd.read_csv("df_train_meta_filtered.csv")
df_val_meta = pd.read_csv("df_val_meta_filtered.csv")

with open("config.json", "r") as f:
    config = json.load(f)
TRAIN_BACK_DIR  = config["TRAIN_BACK_DIR"]
VAL_BACK_DIR    = config["VAL_BACK_DIR"]
COL_DATA_ID     = config["COL_DATA_ID"]
COL_CRYSTAL_SYS = config["COL_CRYSTAL_SYS"]
N_POINTS        = config["N_POINTS"]
RANDOM_SEED     = config["RANDOM_SEED"]

Proses Split Data

In [3]:
BATCH_SIZE_LOAD = 50
SLEEP_BETWEEN   = 1.5
MAX_RETRY       = 3
RETRY_DELAY     = 2.0

def load_intensity_safe(data_id: str, folder: str, n_points: int):
    data_id    = str(data_id)
    extensions = [".csv", ".txt", ".xy", ".dat", ""]

    for ext in extensions:
        fpath = os.path.join(folder, data_id + ext)
        if not os.path.isfile(fpath):
            continue
        for attempt in range(1, MAX_RETRY + 1):
            try:
                df_file = pd.read_csv(fpath, header=0)
                if "y" in df_file.columns:
                    arr = df_file["y"].values
                else:
                    arr = df_file.iloc[:, 0].values
                if len(arr) != n_points:
                    old_idx = np.linspace(0, 1, len(arr))
                    new_idx = np.linspace(0, 1, n_points)
                    arr     = np.interp(new_idx, old_idx, arr)
                return arr
            except Exception as e:
                if attempt < MAX_RETRY:
                    time.sleep(RETRY_DELAY)
                else:
                    print(f"\n   Gagal baca {data_id+ext} setelah {MAX_RETRY}x: {e}")
    return None

def load_dataset_batched(meta_df: pd.DataFrame, folder: str, n_points: int, label: str = "dataset") -> tuple:
    print(f"\n[{label}] Mulai loading ...")
    rows = meta_df.reset_index(drop=True)
    X_list, y_list, ids_list = [], [], []
    for batch_start in tqdm(range(0, len(rows), BATCH_SIZE_LOAD), desc=f"Load [{label}]", unit="batch"):
        batch = rows.iloc[batch_start : batch_start + BATCH_SIZE_LOAD]
        for _, row in batch.iterrows():
            arr = load_intensity_safe(row[COL_DATA_ID], folder, n_points)
            if arr is not None:
                X_list.append(arr)
                y_list.append(row[COL_CRYSTAL_SYS])
                ids_list.append(row[COL_DATA_ID])
    return np.array(X_list), np.array(y_list), ids_list

X_train_all, y_train_all, ids_train = load_dataset_batched(df_train_meta, TRAIN_BACK_DIR, N_POINTS, label="TRAIN")
X_val_raw, y_val, ids_val = load_dataset_batched(df_val_meta, VAL_BACK_DIR, N_POINTS, label="VAL")

# Split: test=50% dari dataset VAL (sesuai aturan 10% di notebook asli Anda dengan seed=42)
val_idx, test_idx = train_test_split(
    np.arange(len(X_val_raw)), test_size=0.50,
    random_state=RANDOM_SEED, stratify=y_val
)

X_train_raw = X_train_all
y_train = y_train_all
X_test_raw = X_val_raw[test_idx]
y_test = y_val[test_idx]
X_val_raw = X_val_raw[val_idx]
y_val = y_val[val_idx]


[TRAIN] Mulai loading ...


Load [TRAIN]: 100%|██████████| 165/165 [01:51<00:00,  1.48batch/s]



[VAL] Mulai loading ...


Load [VAL]: 100%|██████████| 42/42 [00:26<00:00,  1.58batch/s]


In [5]:
np.save("X_train_raw.npy", X_train_raw)
np.save("y_train.npy", y_train)
np.save("X_val_raw.npy", X_val_raw)
np.save("y_val.npy", y_val)
np.save("X_test_raw.npy", X_test_raw)
np.save("y_test.npy", y_test)